# Final Reviewer-Compliant Driver Distraction Personalization Pipeline

This notebook combines and replaces the earlier **WORK 1** and **WORK 1 AUC** notebooks.

It implements:

- State Farm driver-wise global training and personalization
- AUC V2 within-dataset validation using ResNet50, EfficientNet-B0, and MobileNetV3-Small
- True cross-dataset validation: SFD → AUC V2
- Support sizes K = 0, 1, 3, 5, 10, 20
- Head-only, last-block, and full-model adaptation
- Five repeated support-set samplings
- Mean ± standard deviation and 95% confidence intervals
- Matched McNemar tests and paired tests
- Per-class and per-driver metrics
- Image-path and clip/group leakage audits
- Adaptation time per driver
- Latency, FPS, parameter count, and model-size profiling
- Grad-CAM before/after personalization
- ProtoNet metric-learning baseline
- ANIL gradient-based meta-learning baseline
- AUC V1 diagnostic and global-only negative-result reporting
- Canonical master tables to prevent numerical inconsistencies

The notebook intentionally describes the method as a **support-set personalization protocol using standard supervised fine-tuning**, not as a novel meta-learning algorithm.

In [ ]:
# ============================================================
# 1. IMPORTS AND SAFE JUPYTER SETUP
# ============================================================
import os
import gc
import re
import json
import math
import time
import copy
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    confusion_matrix, classification_report
)
from scipy.stats import ttest_rel, binomtest

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", category=UserWarning)

# Force notebook-safe loading. No multiprocessing workers are created.
SAFE_NUM_WORKERS = 0
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("SAFE DataLoader active | num_workers=0")

In [ ]:
# ============================================================
# 2. CONFIGURATION — EDIT PATHS HERE ONLY
# ============================================================
CFG = {
    # State Farm
    "sfd_csv": Path("DATASET/FILES/extracted/SFD/driver_imgs_list.csv"),
    "sfd_root": Path("DATASET/FILES/extracted/SFD/train"),

    # AUC V2 root. The code searches recursively for train/test folders.
    "auc_v2_root": Path("DATASET/FILES/extracted/AUC/v2_cam1_cam2_split_by_driver"),

    # AUC V1
    "auc_v1_root": Path("DATASET/FILES/extracted/AUC/v1_cam1_no_split"),

    "output_dir": Path("FINAL_REVIEWER_OUTPUTS"),
    "image_size": 224,
    "batch_size": 32,
    "support_batch_size": 16,

    "backbones": ["resnet50", "efficientnet_b0", "mobilenet_v3_small"],
    "support_sizes": [0, 1, 3, 5, 10, 20],
    "scopes": ["head", "last_block", "full"],
    "seeds": [11, 22, 33, 44, 55],

    "global_epochs": 20,
    "global_lr": 1e-4,
    "weight_decay": 1e-4,

    "adapt_epochs": {
        "head": 8,
        "last_block": 8,
        "full": 8,
    },
    "adapt_lr": {
        "head": 1e-3,
        "last_block": 5e-5,
        "full": 1e-5,
    },

    "amp": True,
    "debug": False,

    # For a quick syntax test
    "debug_global_epochs": 1,
    "debug_support_sizes": [0, 1],
    "debug_seeds": [11],
    "debug_scopes": ["head"],
}

CFG["output_dir"].mkdir(parents=True, exist_ok=True)
for sub in ["tables", "figures", "checkpoints", "predictions", "audits", "gradcam"]:
    (CFG["output_dir"] / sub).mkdir(exist_ok=True)

print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in CFG.items()}, indent=2))

In [ ]:
# ============================================================
# 3. REPRODUCIBILITY AND CLASS MAPPING
# ============================================================
SFD_CLASS_NAMES = {
    0: "safe_driving",
    1: "texting_right",
    2: "phone_right",
    3: "texting_left",
    4: "phone_left",
    5: "radio",
    6: "drinking",
    7: "reaching_behind",
    8: "hair_makeup",
    9: "talking_passenger",
}

AUC_FOLDER_TO_CANONICAL = {
    "c0": "safe_driving", "safe driving": "safe_driving", "drive safe": "safe_driving",
    "c1": "texting_right", "text right": "texting_right",
    "c2": "phone_right", "talk right": "phone_right",
    "c3": "texting_left", "text left": "texting_left",
    "c4": "phone_left", "talk left": "phone_left",
    "c5": "radio", "adjust radio": "radio",
    "c6": "drinking", "drink": "drinking",
    "c7": "reaching_behind", "reach behind": "reaching_behind",
    "c8": "hair_makeup", "hair & makeup": "hair_makeup", "hair and makeup": "hair_makeup",
    "c9": "talking_passenger", "talk passenger": "talking_passenger",
}

CANONICAL_TO_LABEL = {name: idx for idx, name in SFD_CLASS_NAMES.items()}

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(CFG["seeds"][0])

In [ ]:
# ============================================================
# 4. TRANSFORMS
# ============================================================
IMG_SIZE = CFG["image_size"]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(7),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# Horizontal flip is intentionally omitted because left/right actions are distinct classes.
eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [ ]:
# ============================================================
# 5. GENERIC DATASET AND SAFE DATALOADER
# ============================================================
class ImageFrameDataset(Dataset):
    def __init__(self, df, transform=None, return_meta=False):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.return_meta = return_meta

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = Path(row["image_path"])
        try:
            image = Image.open(path).convert("RGB")
        except (UnidentifiedImageError, OSError) as exc:
            raise RuntimeError(f"Could not read image: {path}") from exc

        if self.transform:
            image = self.transform(image)

        label = int(row["label"])
        if self.return_meta:
            meta = {
                "image_path": str(path),
                "driver": str(row.get("driver", "")),
                "group_id": str(row.get("group_id", "")),
                "class_name": str(row.get("class_name", SFD_CLASS_NAMES[label])),
            }
            return image, label, meta
        return image, label

def make_loader(df, transform, shuffle=False, return_meta=False, batch_size=None):
    ds = ImageFrameDataset(df, transform=transform, return_meta=return_meta)
    return DataLoader(
        ds,
        batch_size=batch_size or CFG["batch_size"],
        shuffle=shuffle,
        num_workers=0,
        pin_memory=False,
        persistent_workers=False,
        drop_last=False,
    )

In [ ]:
# ============================================================
# 6. STATE FARM INDEX AND DRIVER-WISE SPLIT
# ============================================================
def load_sfd_index():
    csv_path = CFG["sfd_csv"]
    root = CFG["sfd_root"]
    if not csv_path.exists():
        raise FileNotFoundError(f"State Farm CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    required = {"subject", "classname", "img"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing SFD CSV columns: {missing}")

    df["label"] = df["classname"].str.replace("c", "", regex=False).astype(int)
    df["class_name"] = df["label"].map(SFD_CLASS_NAMES)
    df["driver"] = df["subject"].astype(str)
    df["image_path"] = df.apply(
        lambda r: str(root / r["classname"] / r["img"]), axis=1
    )

    # State Farm has no official clip ID. A conservative pseudo-group is derived
    # from filename stem to support adjacency audits, but it must not be claimed
    # as an official video identifier.
    df["frame_number"] = df["img"].astype(str).str.extract(r"(\d+)", expand=False)
    df["frame_number"] = pd.to_numeric(df["frame_number"], errors="coerce")
    df["group_id"] = df["driver"] + "_" + df["classname"]

    exists = df["image_path"].map(lambda p: Path(p).exists())
    if not exists.all():
        print("Warning: missing SFD images:", int((~exists).sum()))
        df = df[exists].reset_index(drop=True)

    drivers = sorted(df["driver"].unique())
    # Preserve the common fixed split if available.
    preferred_test = ["p052", "p056", "p061", "p064", "p066"]
    test_drivers = [d for d in preferred_test if d in drivers]
    if len(test_drivers) < 5:
        test_drivers = drivers[-5:]

    train_df = df[~df["driver"].isin(test_drivers)].reset_index(drop=True)
    test_df = df[df["driver"].isin(test_drivers)].reset_index(drop=True)

    assert set(train_df.driver).isdisjoint(set(test_df.driver))
    print("SFD train:", train_df.shape, "drivers:", train_df.driver.nunique())
    print("SFD test :", test_df.shape, "drivers:", test_df.driver.nunique())
    print("SFD test drivers:", sorted(test_df.driver.unique()))
    return train_df, test_df

sfd_train_df, sfd_test_df = load_sfd_index()

In [ ]:
# ============================================================
# 7. AUC V2 INDEX BUILDER
# ============================================================
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def infer_canonical_class(folder_name):
    key = folder_name.strip().lower()
    if key in AUC_FOLDER_TO_CANONICAL:
        return AUC_FOLDER_TO_CANONICAL[key]
    if re.fullmatch(r"c\d+", key):
        return SFD_CLASS_NAMES[int(key[1:])]
    raise ValueError(f"Unknown class folder: {folder_name}")

def find_split_dirs(root):
    root = Path(root)
    candidates = []
    for train_dir in root.rglob("train"):
        if train_dir.is_dir():
            test_dir = train_dir.parent / "test"
            if test_dir.exists():
                candidates.append((train_dir, test_dir))
    if not candidates:
        raise FileNotFoundError(f"No train/test pair found under {root}")
    print("AUC V2 split pairs found:", len(candidates))
    return candidates

def parse_auc_driver_and_group(path, split_root):
    rel = Path(path).relative_to(split_root)
    parts = list(rel.parts)
    class_folder = parts[0] if parts else ""
    filename = Path(path).stem

    # Try to infer driver from folder hierarchy or filename.
    driver = ""
    for part in parts[1:-1]:
        if re.search(r"(driver|subject|person|p\d+)", part, flags=re.I):
            driver = part
            break
    if not driver:
        m = re.search(r"(?:driver|subject|person|p)[_-]?(\d+)", filename, flags=re.I)
        driver = f"driver_{m.group(1)}" if m else "unknown_driver"

    # Group/clip inference. This is an audit aid; verify against dataset metadata.
    group = filename
    group = re.sub(r"([_-]?(frame|img|image)?[_-]?\d+)$", "", group, flags=re.I)
    if not group:
        group = Path(path).parent.name
    group_id = f"{driver}_{class_folder}_{group}"
    return driver, group_id

def build_auc_v2_index():
    rows = []
    for camera_id, (train_dir, test_dir) in enumerate(find_split_dirs(CFG["auc_v2_root"]), start=1):
        for split, split_dir in [("train", train_dir), ("test", test_dir)]:
            for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
                try:
                    class_name = infer_canonical_class(class_dir.name)
                except ValueError:
                    continue
                label = CANONICAL_TO_LABEL[class_name]
                for p in class_dir.rglob("*"):
                    if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                        driver, group_id = parse_auc_driver_and_group(p, split_dir)
                        rows.append({
                            "dataset": "AUC_V2",
                            "camera": camera_id,
                            "split": split,
                            "image_path": str(p),
                            "label": label,
                            "class_name": class_name,
                            "driver": driver,
                            "group_id": group_id,
                        })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No AUC V2 images were indexed.")
    print(df.groupby(["split", "class_name"]).size())
    print("AUC V2 drivers:", df.driver.nunique())
    return df[df.split=="train"].reset_index(drop=True), df[df.split=="test"].reset_index(drop=True)

auc_v2_train_df, auc_v2_test_df = build_auc_v2_index()

In [ ]:
# ============================================================
# 8. MODEL FACTORY AND ADAPTATION SCOPES
# ============================================================
def build_model(backbone, num_classes=10, pretrained=True):
    if backbone == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif backbone == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif backbone == "mobilenet_v3_small":
        weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.mobilenet_v3_small(weights=weights)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    else:
        raise ValueError(backbone)
    return model

def get_classifier(model, backbone):
    if backbone == "resnet50":
        return model.fc
    return model.classifier

def set_adaptation_scope(model, backbone, scope):
    for p in model.parameters():
        p.requires_grad = False

    if scope == "head":
        for p in get_classifier(model, backbone).parameters():
            p.requires_grad = True

    elif scope == "last_block":
        if backbone == "resnet50":
            modules = [model.layer4, model.fc]
        elif backbone == "efficientnet_b0":
            modules = [model.features[-1], model.classifier]
        elif backbone == "mobilenet_v3_small":
            modules = [model.features[-1], model.classifier]
        for module in modules:
            for p in module.parameters():
                p.requires_grad = True

    elif scope == "full":
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError(scope)

    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# ============================================================
# 9. TRAINING, EVALUATION, CHECKPOINTING
# ============================================================
def train_one_epoch(model, loader, optimizer, scaler=None):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss, total, correct = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        use_amp = CFG["amp"] and DEVICE.type == "cuda"
        with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        if scaler is not None and use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
    }

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, preds, probs, metas = [], [], [], []

    for batch in loader:
        if len(batch) == 3:
            images, labels, meta = batch
        else:
            images, labels = batch
            meta = None

        images = images.to(DEVICE)
        logits = model(images)
        prob = torch.softmax(logits, dim=1).cpu().numpy()
        pred = prob.argmax(1)

        ys.extend(labels.numpy().tolist())
        preds.extend(pred.tolist())
        probs.extend(prob.tolist())

        if meta is not None:
            n = len(labels)
            for i in range(n):
                metas.append({k: meta[k][i] for k in meta})

    return {
        "y_true": np.asarray(ys),
        "y_pred": np.asarray(preds),
        "probs": np.asarray(probs),
        "meta": metas,
        "accuracy": accuracy_score(ys, preds),
        "macro_f1": f1_score(ys, preds, average="macro", zero_division=0),
    }

def train_global_model(dataset_name, backbone, train_df):
    ckpt = CFG["output_dir"] / "checkpoints" / f"{dataset_name}_{backbone}_global.pt"
    hist_path = CFG["output_dir"] / "tables" / f"{dataset_name}_{backbone}_global_history.csv"

    if ckpt.exists():
        model = build_model(backbone, pretrained=False)
        model.load_state_dict(torch.load(ckpt, map_location="cpu"))
        return model.to(DEVICE)

    model = build_model(backbone, pretrained=True).to(DEVICE)
    loader = make_loader(train_df, train_tfms, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["global_lr"],
                                  weight_decay=CFG["weight_decay"])
    scaler = torch.cuda.amp.GradScaler(enabled=(CFG["amp"] and DEVICE.type=="cuda"))

    epochs = CFG["debug_global_epochs"] if CFG["debug"] else CFG["global_epochs"]
    history = []
    for epoch in range(1, epochs + 1):
        metrics = train_one_epoch(model, loader, optimizer, scaler)
        metrics["epoch"] = epoch
        history.append(metrics)
        print(dataset_name, backbone, epoch, metrics)

    torch.save(model.state_dict(), ckpt)
    pd.DataFrame(history).to_csv(hist_path, index=False)
    return model

In [ ]:
# ============================================================
# 10. LEAKAGE-SAFE SUPPORT / QUERY CONSTRUCTION
# ============================================================
def support_query_split(driver_df, k, seed, strict_group=True):
    if k == 0:
        return driver_df.iloc[0:0].copy(), driver_df.copy()

    rng = np.random.default_rng(seed)
    support_parts, query_parts = [], []

    for label, cls_df in driver_df.groupby("label"):
        cls_df = cls_df.copy().reset_index(drop=True)

        if strict_group and cls_df["group_id"].nunique() >= 2:
            groups = cls_df["group_id"].drop_duplicates().to_numpy()
            rng.shuffle(groups)

            support_groups, chosen = [], 0
            for g in groups:
                support_groups.append(g)
                chosen += int((cls_df.group_id == g).sum())
                if chosen >= k:
                    break

            support_cls = cls_df[cls_df.group_id.isin(support_groups)]
            # Restrict to exactly K where possible.
            support_cls = support_cls.sample(
                n=min(k, len(support_cls)), random_state=seed
            )
            query_cls = cls_df[~cls_df.group_id.isin(support_groups)]
        else:
            idx = np.arange(len(cls_df))
            rng.shuffle(idx)
            n_support = min(k, max(0, len(cls_df)-1))
            support_cls = cls_df.iloc[idx[:n_support]]
            query_cls = cls_df.iloc[idx[n_support:]]

        support_parts.append(support_cls)
        query_parts.append(query_cls)

    support = pd.concat(support_parts, ignore_index=True)
    query = pd.concat(query_parts, ignore_index=True)

    # Hard image-level leakage assertion.
    overlap = set(support.image_path) & set(query.image_path)
    assert not overlap, f"Image leakage detected: {len(overlap)} files"

    return support, query

def leakage_audit(support, query, dataset, driver, k, seed):
    image_overlap = len(set(support.image_path) & set(query.image_path))
    group_overlap = len(set(support.group_id) & set(query.group_id))
    return {
        "dataset": dataset, "driver": driver, "k": k, "seed": seed,
        "support_n": len(support), "query_n": len(query),
        "image_path_overlap": image_overlap,
        "group_id_overlap": group_overlap,
        "status": "PASS" if image_overlap == 0 and group_overlap == 0 else "CHECK",
    }

In [ ]:
# ============================================================
# 11. DRIVER PERSONALIZATION EXPERIMENT
# ============================================================
def prediction_frame(result, dataset, backbone, k, seed, scope, phase, driver):
    meta = pd.DataFrame(result["meta"])
    out = meta.copy()
    out["dataset"] = dataset
    out["backbone"] = backbone
    out["k"] = k
    out["seed"] = seed
    out["scope"] = scope
    out["phase"] = phase
    out["driver_scope"] = driver
    out["y_true"] = result["y_true"]
    out["y_pred"] = result["y_pred"]
    out["true_class"] = [SFD_CLASS_NAMES[int(x)] for x in result["y_true"]]
    out["pred_class"] = [SFD_CLASS_NAMES[int(x)] for x in result["y_pred"]]
    out["correct"] = (out.y_true == out.y_pred).astype(int)
    return out

def adapt_for_driver(global_model, backbone, support_df, scope, seed):
    set_seed(seed)
    model = copy.deepcopy(global_model).to(DEVICE)
    trainable_params = set_adaptation_scope(model, backbone, scope)

    loader = make_loader(
        support_df, train_tfms, shuffle=True,
        batch_size=CFG["support_batch_size"]
    )
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=CFG["adapt_lr"][scope],
        weight_decay=CFG["weight_decay"],
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(CFG["amp"] and DEVICE.type=="cuda"))

    start = time.perf_counter()
    for _ in range(CFG["adapt_epochs"][scope]):
        train_one_epoch(model, loader, optimizer, scaler)
    elapsed = time.perf_counter() - start
    return model, elapsed, trainable_params

def run_personalization_experiment(dataset_name, train_df, test_df, backbones):
    support_sizes = CFG["debug_support_sizes"] if CFG["debug"] else CFG["support_sizes"]
    seeds = CFG["debug_seeds"] if CFG["debug"] else CFG["seeds"]
    scopes = CFG["debug_scopes"] if CFG["debug"] else CFG["scopes"]

    all_predictions, timing_rows, audit_rows = [], [], []

    for backbone in backbones:
        print("\nGLOBAL TRAINING:", dataset_name, backbone)
        global_model = train_global_model(dataset_name, backbone, train_df)

        for k in support_sizes:
            for seed in seeds:
                for driver, driver_df in test_df.groupby("driver"):
                    support_df, query_df = support_query_split(
                        driver_df.reset_index(drop=True), k, seed, strict_group=True
                    )
                    audit_rows.append(leakage_audit(
                        support_df, query_df, dataset_name, driver, k, seed
                    ))

                    q_loader = make_loader(query_df, eval_tfms, return_meta=True)

                    before = evaluate(global_model, q_loader)
                    all_predictions.append(prediction_frame(
                        before, dataset_name, backbone, k, seed, "none", "before", driver
                    ))

                    if k == 0 or len(support_df) == 0:
                        continue

                    for scope in scopes:
                        adapted, seconds, trainable = adapt_for_driver(
                            global_model, backbone, support_df, scope, seed
                        )
                        after = evaluate(adapted, q_loader)
                        all_predictions.append(prediction_frame(
                            after, dataset_name, backbone, k, seed, scope, "after", driver
                        ))
                        timing_rows.append({
                            "dataset": dataset_name, "backbone": backbone,
                            "driver": driver, "k": k, "seed": seed, "scope": scope,
                            "adaptation_time_seconds": seconds,
                            "trainable_parameters": trainable,
                        })
                        del adapted
                        gc.collect()
                        if DEVICE.type == "cuda":
                            torch.cuda.empty_cache()

    pred_df = pd.concat(all_predictions, ignore_index=True)
    timing_df = pd.DataFrame(timing_rows)
    audit_df = pd.DataFrame(audit_rows)

    pred_df.to_csv(CFG["output_dir"]/ "predictions" / f"{dataset_name}_all_predictions.csv", index=False)
    timing_df.to_csv(CFG["output_dir"]/ "tables" / f"{dataset_name}_adaptation_time_per_driver.csv", index=False)
    audit_df.to_csv(CFG["output_dir"]/ "audits" / f"{dataset_name}_support_query_leakage_audit.csv", index=False)

    return pred_df, timing_df, audit_df

In [ ]:
# ============================================================
# 12. STATISTICS, PER-CLASS, PER-DRIVER, AND CANONICAL TABLES
# ============================================================
def ci95(values):
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return np.nan, np.nan
    mean = values.mean()
    margin = 1.96 * values.std(ddof=1) / math.sqrt(len(values))
    return mean - margin, mean + margin

def aggregate_results(pred_df):
    rows = []
    group_cols = ["dataset", "backbone", "k", "scope", "phase"]

    for keys, g in pred_df.groupby(group_cols):
        seed_metrics = []
        for seed, sg in g.groupby("seed"):
            seed_metrics.append({
                "seed": seed,
                "accuracy": accuracy_score(sg.y_true, sg.y_pred),
                "macro_f1": f1_score(sg.y_true, sg.y_pred, average="macro", zero_division=0),
            })
        sm = pd.DataFrame(seed_metrics)
        acc_lo, acc_hi = ci95(sm.accuracy)
        f1_lo, f1_hi = ci95(sm.macro_f1)
        row = dict(zip(group_cols, keys))
        row.update({
            "n_seeds": len(sm),
            "accuracy_mean": sm.accuracy.mean(),
            "accuracy_std": sm.accuracy.std(ddof=1),
            "accuracy_ci95_low": acc_lo,
            "accuracy_ci95_high": acc_hi,
            "macro_f1_mean": sm.macro_f1.mean(),
            "macro_f1_std": sm.macro_f1.std(ddof=1),
            "macro_f1_ci95_low": f1_lo,
            "macro_f1_ci95_high": f1_hi,
        })
        rows.append(row)
    return pd.DataFrame(rows)

def per_class_metrics(pred_df):
    rows = []
    cols = ["dataset","backbone","k","seed","scope","phase"]
    for keys, g in pred_df.groupby(cols):
        p, r, f, s = precision_recall_fscore_support(
            g.y_true, g.y_pred, labels=list(range(10)), zero_division=0
        )
        for label in range(10):
            row = dict(zip(cols, keys))
            row.update({
                "label": label, "class_name": SFD_CLASS_NAMES[label],
                "precision": p[label], "recall": r[label],
                "f1": f[label], "support": s[label],
            })
            rows.append(row)
    return pd.DataFrame(rows)

def per_driver_metrics(pred_df):
    rows = []
    cols = ["dataset","backbone","k","seed","scope","phase","driver_scope"]
    for keys, g in pred_df.groupby(cols):
        row = dict(zip(cols, keys))
        row.update({
            "n": len(g),
            "accuracy": accuracy_score(g.y_true, g.y_pred),
            "macro_f1": f1_score(g.y_true, g.y_pred, average="macro", zero_division=0),
        })
        rows.append(row)
    return pd.DataFrame(rows)

def matched_tests(pred_df):
    rows_t, rows_m = [], []
    after = pred_df[pred_df.phase=="after"]

    for (dataset, backbone, k, seed, scope), ag in after.groupby(
        ["dataset","backbone","k","seed","scope"]
    ):
        bg = pred_df[
            (pred_df.dataset==dataset) &
            (pred_df.backbone==backbone) &
            (pred_df.k==k) &
            (pred_df.seed==seed) &
            (pred_df.phase=="before")
        ]
        merged = bg.merge(
            ag, on=["image_path","driver_scope","y_true"],
            suffixes=("_before","_after")
        )
        if merged.empty:
            continue

        before_correct = (merged.y_pred_before == merged.y_true).astype(int)
        after_correct = (merged.y_pred_after == merged.y_true).astype(int)

        b = int(((before_correct==1) & (after_correct==0)).sum())
        c = int(((before_correct==0) & (after_correct==1)).sum())
        p_mcnemar = binomtest(min(b,c), n=b+c, p=0.5).pvalue if b+c else 1.0

        t_stat, p_t = ttest_rel(after_correct, before_correct)

        base = {
            "dataset": dataset, "backbone": backbone,
            "k": k, "seed": seed, "scope": scope, "n_matched": len(merged)
        }
        rows_m.append({**base, "before_correct_after_wrong": b,
                       "before_wrong_after_correct": c,
                       "mcnemar_exact_p": p_mcnemar})
        rows_t.append({**base, "paired_t_stat": t_stat, "paired_t_p": p_t,
                       "mean_before": before_correct.mean(),
                       "mean_after": after_correct.mean()})
    return pd.DataFrame(rows_t), pd.DataFrame(rows_m)

def save_all_analysis(dataset_name, pred_df):
    summary = aggregate_results(pred_df)
    per_class = per_class_metrics(pred_df)
    per_driver = per_driver_metrics(pred_df)
    paired, mcnemar = matched_tests(pred_df)

    base = CFG["output_dir"] / "tables"
    summary.to_csv(base/f"{dataset_name}_mean_std_ci95_summary.csv", index=False)
    per_class.to_csv(base/f"{dataset_name}_per_class_metrics.csv", index=False)
    per_driver.to_csv(base/f"{dataset_name}_per_driver_metrics.csv", index=False)
    paired.to_csv(base/f"{dataset_name}_paired_tests.csv", index=False)
    mcnemar.to_csv(base/f"{dataset_name}_mcnemar_tests.csv", index=False)
    return summary, per_class, per_driver, paired, mcnemar

## 13. Run State Farm Experiment

Set `CFG["debug"] = True` first for a short verification run.  
For the final reviewer tables, use `CFG["debug"] = False`.

In [ ]:
# SFD full reviewer experiment
# sfd_pred, sfd_timing, sfd_audit = run_personalization_experiment(
#     "SFD", sfd_train_df, sfd_test_df, CFG["backbones"]
# )
# sfd_summary, sfd_class, sfd_driver, sfd_paired, sfd_mcnemar = save_all_analysis(
#     "SFD", sfd_pred
# )

## 14. Run AUC V2 Within-Dataset Multi-Backbone Experiment

This directly addresses the reviewer request to extend AUC V2 beyond ResNet50.

In [ ]:
# AUC V2 within-dataset training and personalization
# aucv2_pred, aucv2_timing, aucv2_audit = run_personalization_experiment(
#     "AUCV2_WITHIN", auc_v2_train_df, auc_v2_test_df, CFG["backbones"]
# )
# aucv2_summary, aucv2_class, aucv2_driver, aucv2_paired, aucv2_mcnemar = save_all_analysis(
#     "AUCV2_WITHIN", aucv2_pred
# )

In [ ]:
# ============================================================
# 15. TRUE CROSS-DATASET: TRAIN ON SFD, ADAPT/TEST ON AUC V2
# ============================================================
def run_cross_dataset_sfd_to_aucv2():
    all_predictions, timing_rows, audit_rows = [], [], []
    support_sizes = CFG["debug_support_sizes"] if CFG["debug"] else CFG["support_sizes"]
    seeds = CFG["debug_seeds"] if CFG["debug"] else CFG["seeds"]
    scopes = CFG["debug_scopes"] if CFG["debug"] else CFG["scopes"]

    for backbone in CFG["backbones"]:
        global_model = train_global_model("SFD", backbone, sfd_train_df)

        # If AUC V2 driver IDs are unavailable, use one pooled target domain.
        groups = list(auc_v2_test_df.groupby("driver"))
        if len(groups) == 1 and groups[0][0] == "unknown_driver":
            groups = [("AUCV2_target", auc_v2_test_df)]

        for k in support_sizes:
            for seed in seeds:
                for driver, driver_df in groups:
                    support_df, query_df = support_query_split(
                        driver_df.reset_index(drop=True), k, seed, strict_group=True
                    )
                    audit_rows.append(leakage_audit(
                        support_df, query_df, "SFD_TO_AUCV2", driver, k, seed
                    ))

                    q_loader = make_loader(query_df, eval_tfms, return_meta=True)
                    before = evaluate(global_model, q_loader)
                    all_predictions.append(prediction_frame(
                        before, "SFD_TO_AUCV2", backbone, k, seed,
                        "none", "before", driver
                    ))

                    if k == 0 or support_df.empty:
                        continue

                    for scope in scopes:
                        adapted, seconds, trainable = adapt_for_driver(
                            global_model, backbone, support_df, scope, seed
                        )
                        after = evaluate(adapted, q_loader)
                        all_predictions.append(prediction_frame(
                            after, "SFD_TO_AUCV2", backbone, k, seed,
                            scope, "after", driver
                        ))
                        timing_rows.append({
                            "dataset": "SFD_TO_AUCV2", "backbone": backbone,
                            "driver": driver, "k": k, "seed": seed, "scope": scope,
                            "adaptation_time_seconds": seconds,
                            "trainable_parameters": trainable,
                        })
                        del adapted
                        gc.collect()
                        if DEVICE.type == "cuda":
                            torch.cuda.empty_cache()

    pred_df = pd.concat(all_predictions, ignore_index=True)
    timing_df = pd.DataFrame(timing_rows)
    audit_df = pd.DataFrame(audit_rows)

    pred_df.to_csv(CFG["output_dir"]/ "predictions" / "SFD_TO_AUCV2_all_predictions.csv", index=False)
    timing_df.to_csv(CFG["output_dir"]/ "tables" / "SFD_TO_AUCV2_adaptation_time.csv", index=False)
    audit_df.to_csv(CFG["output_dir"]/ "audits" / "SFD_TO_AUCV2_leakage_audit.csv", index=False)
    save_all_analysis("SFD_TO_AUCV2", pred_df)
    return pred_df

# cross_pred = run_cross_dataset_sfd_to_aucv2()

In [ ]:
# ============================================================
# 16. AUC V1 DIAGNOSTIC AND GLOBAL-ONLY NEGATIVE RESULT
# ============================================================
def build_auc_v1_index():
    root = CFG["auc_v1_root"]
    train_csv = root / "Train_data_list.csv"
    test_csv = root / "Test_data_list.csv"

    if not train_csv.exists() or not test_csv.exists():
        raise FileNotFoundError("AUC V1 Train_data_list.csv/Test_data_list.csv not found.")

    def image_col(df):
        best, best_count = None, -1
        for c in df.columns:
            count = df[c].dropna().astype(str).str.lower().str.contains(
                r"\.(?:jpg|jpeg|png|bmp|webp)", regex=True
            ).sum()
            if count > best_count:
                best, best_count = c, count
        return best

    def resolve(raw):
        p = root / str(raw)
        if p.exists():
            return p
        matches = list(root.rglob(Path(str(raw)).name))
        return matches[0] if matches else p

    rows = []
    raw_train, raw_test = pd.read_csv(train_csv), pd.read_csv(test_csv)
    for split, raw in [("train", raw_train), ("test", raw_test)]:
        col = image_col(raw)
        for _, r in raw.iterrows():
            p = resolve(r[col])
            parent = p.parent.name.lower()
            cname = AUC_FOLDER_TO_CANONICAL.get(parent)
            if cname is None:
                continue
            rows.append({
                "dataset": "AUC_V1", "split": split,
                "image_path": str(p), "label": CANONICAL_TO_LABEL[cname],
                "class_name": cname, "driver": "unknown",
                "group_id": p.stem,
            })

    df = pd.DataFrame(rows)
    driver_cols = [c for c in raw_train.columns if re.search(r"driver|subject|participant", c, re.I)]
    diagnostic = pd.DataFrame([{
        "dataset": "AUC_V1",
        "n_train": int((df.split=="train").sum()),
        "n_test": int((df.split=="test").sum()),
        "n_total": len(df),
        "n_classes": df.class_name.nunique(),
        "has_driver_or_subject_column": bool(driver_cols),
        "status": "not_subject_wise_personalization_ready" if not driver_cols else "check_driver_metadata",
        "reason": "No reliable driver/subject identifiers for driver-wise support/query construction."
    }])
    diagnostic.to_csv(CFG["output_dir"]/ "audits" / "AUC_V1_negative_result_diagnostic.csv", index=False)
    return df[df.split=="train"].reset_index(drop=True), df[df.split=="test"].reset_index(drop=True), diagnostic

def evaluate_auc_v1_global_only():
    train_df, test_df, diagnostic = build_auc_v1_index()
    rows = []
    for backbone in CFG["backbones"]:
        model = train_global_model("AUCV1_GLOBAL", backbone, train_df)
        result = evaluate(model, make_loader(test_df, eval_tfms, return_meta=True))
        rows.append({
            "dataset": "AUC_V1",
            "backbone": backbone,
            "setting": "global_only",
            "accuracy": result["accuracy"],
            "macro_f1": result["macro_f1"],
            "personalization_result": "Not performed",
            "reason": diagnostic.iloc[0]["reason"],
        })
    out = pd.DataFrame(rows)
    out.to_csv(CFG["output_dir"]/ "tables" / "AUC_V1_global_negative_result.csv", index=False)
    return out

# auc_v1_global_result = evaluate_auc_v1_global_only()

In [ ]:
# ============================================================
# 17. PROTONET BASELINE
# ============================================================
class FeatureExtractor(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        model = build_model(backbone, pretrained=True)
        if backbone == "resnet50":
            self.features = nn.Sequential(*list(model.children())[:-1])
            self.out_dim = model.fc.in_features
        elif backbone == "efficientnet_b0":
            self.features = nn.Sequential(model.features, model.avgpool)
            self.out_dim = model.classifier[1].in_features
        else:
            self.features = nn.Sequential(model.features, model.avgpool)
            self.out_dim = model.classifier[3].in_features

    def forward(self, x):
        return self.features(x).flatten(1)

def protonet_predict(extractor, support_loader, query_loader):
    extractor.eval()
    support_features, support_labels = [], []
    with torch.no_grad():
        for x, y in support_loader:
            support_features.append(extractor(x.to(DEVICE)).cpu())
            support_labels.append(y)
    z = torch.cat(support_features)
    y = torch.cat(support_labels)

    prototypes = []
    classes = sorted(y.unique().tolist())
    for c in classes:
        prototypes.append(z[y==c].mean(0))
    prototypes = torch.stack(prototypes).to(DEVICE)

    ys, preds = [], []
    with torch.no_grad():
        for x, yq in query_loader:
            q = extractor(x.to(DEVICE))
            distances = torch.cdist(q, prototypes)
            pred_idx = distances.argmin(1).cpu()
            pred = torch.tensor([classes[i] for i in pred_idx.tolist()])
            ys.extend(yq.tolist())
            preds.extend(pred.tolist())
    return np.asarray(ys), np.asarray(preds)

def run_protonet_baseline(dataset_name, test_df, backbone="resnet50"):
    rows = []
    extractor = FeatureExtractor(backbone).to(DEVICE)

    for k in [1,3,5,10,20]:
        for seed in CFG["seeds"]:
            ys_all, ps_all = [], []
            for driver, driver_df in test_df.groupby("driver"):
                support, query = support_query_split(driver_df, k, seed, strict_group=True)
                if support.empty or query.empty:
                    continue
                ys, ps = protonet_predict(
                    extractor,
                    make_loader(support, eval_tfms, batch_size=CFG["support_batch_size"]),
                    make_loader(query, eval_tfms)
                )
                ys_all.extend(ys); ps_all.extend(ps)
            rows.append({
                "dataset": dataset_name, "method": "ProtoNet",
                "backbone": backbone, "k": k, "seed": seed,
                "accuracy": accuracy_score(ys_all, ps_all),
                "macro_f1": f1_score(ys_all, ps_all, average="macro", zero_division=0)
            })
    df = pd.DataFrame(rows)
    df.to_csv(CFG["output_dir"]/ "tables" / f"{dataset_name}_protonet_baseline.csv", index=False)
    return df

# protonet_sfd = run_protonet_baseline("SFD", sfd_test_df)

In [ ]:
# ============================================================
# 18. ANIL BASELINE (GRADIENT-BASED META-LEARNING)
# ============================================================
class ANILHead(nn.Module):
    def __init__(self, feature_dim, num_classes=10):
        super().__init__()
        self.linear = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

def anil_adapt_and_predict(extractor, support_loader, query_loader,
                           inner_steps=5, inner_lr=1e-2):
    extractor.eval()
    feature_dim = extractor.out_dim
    head = ANILHead(feature_dim, 10).to(DEVICE)
    optimizer = torch.optim.SGD(head.parameters(), lr=inner_lr)

    for _ in range(inner_steps):
        for x, y in support_loader:
            with torch.no_grad():
                features = extractor(x.to(DEVICE))
            logits = head(features)
            loss = F.cross_entropy(logits, y.to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    ys, preds = [], []
    with torch.no_grad():
        for x, y in query_loader:
            features = extractor(x.to(DEVICE))
            pred = head(features).argmax(1).cpu()
            ys.extend(y.tolist())
            preds.extend(pred.tolist())
    return np.asarray(ys), np.asarray(preds)

def run_anil_baseline(dataset_name, test_df, backbone="resnet50"):
    rows = []
    extractor = FeatureExtractor(backbone).to(DEVICE)

    for k in [1,3,5,10,20]:
        for seed in CFG["seeds"]:
            ys_all, ps_all = [], []
            for driver, driver_df in test_df.groupby("driver"):
                support, query = support_query_split(driver_df, k, seed, strict_group=True)
                if support.empty or query.empty:
                    continue
                ys, ps = anil_adapt_and_predict(
                    extractor,
                    make_loader(support, eval_tfms, shuffle=True,
                                batch_size=CFG["support_batch_size"]),
                    make_loader(query, eval_tfms)
                )
                ys_all.extend(ys); ps_all.extend(ps)

            rows.append({
                "dataset": dataset_name, "method": "ANIL",
                "backbone": backbone, "k": k, "seed": seed,
                "accuracy": accuracy_score(ys_all, ps_all),
                "macro_f1": f1_score(ys_all, ps_all, average="macro", zero_division=0)
            })
    df = pd.DataFrame(rows)
    df.to_csv(CFG["output_dir"]/ "tables" / f"{dataset_name}_ANIL_baseline.csv", index=False)
    return df

# anil_sfd = run_anil_baseline("SFD", sfd_test_df)

In [ ]:
# ============================================================
# 19. LATENCY, FPS, PARAMETER COUNT, MODEL SIZE
# ============================================================
def profile_model(model, backbone, repeats=100):
    model = model.to(DEVICE).eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    with torch.no_grad():
        for _ in range(10):
            _ = model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()
        for _ in range(repeats):
            _ = model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

    latency_ms = elapsed / repeats * 1000
    fps = 1000 / latency_ms
    params = sum(p.numel() for p in model.parameters())

    tmp = CFG["output_dir"]/ "checkpoints" / f"_profile_{backbone}.pt"
    torch.save(model.state_dict(), tmp)
    size_mb = tmp.stat().st_size / (1024**2)
    tmp.unlink(missing_ok=True)

    return {
        "backbone": backbone,
        "device": str(DEVICE),
        "parameters": params,
        "model_size_mb": size_mb,
        "latency_ms_batch1": latency_ms,
        "fps_batch1": fps,
        "fp16_tested": False,
        "int8_tested": False,
        "real_edge_device_tested": False,
    }

def create_deployment_table():
    rows = []
    for backbone in CFG["backbones"]:
        model = build_model(backbone, pretrained=False)
        ckpt = CFG["output_dir"]/ "checkpoints" / f"SFD_{backbone}_global.pt"
        if ckpt.exists():
            model.load_state_dict(torch.load(ckpt, map_location="cpu"))
        rows.append(profile_model(model, backbone))
    df = pd.DataFrame(rows)
    df.to_csv(CFG["output_dir"]/ "tables" / "latency_fps_model_size.csv", index=False)
    return df

# deployment_table = create_deployment_table()

In [ ]:
# ============================================================
# 20. GRAD-CAM BEFORE / AFTER PERSONALIZATION
# ============================================================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._forward_hook)
        target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inp, out):
        self.activations = out.detach()

    def _backward_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, image_tensor, class_idx=None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(image_tensor)
        if class_idx is None:
            class_idx = logits.argmax(1).item()
        logits[:, class_idx].sum().backward()

        weights = self.gradients.mean(dim=(2,3), keepdim=True)
        cam = (weights * self.activations).sum(1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, image_tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

def gradcam_target_layer(model, backbone):
    if backbone == "resnet50":
        return model.layer4[-1]
    if backbone == "efficientnet_b0":
        return model.features[-1]
    return model.features[-1]

def generate_gradcam_pair(global_model, personalized_model, backbone, image_path, true_label):
    import matplotlib.pyplot as plt

    pil = Image.open(image_path).convert("RGB")
    x = eval_tfms(pil).unsqueeze(0).to(DEVICE)

    gc_before = GradCAM(global_model, gradcam_target_layer(global_model, backbone))
    gc_after = GradCAM(personalized_model, gradcam_target_layer(personalized_model, backbone))

    cam_b, pred_b = gc_before(x)
    cam_a, pred_a = gc_after(x)

    fig, axes = plt.subplots(1, 3, figsize=(12,4))
    axes[0].imshow(pil); axes[0].axis("off")
    axes[0].set_title(f"Input | True: {SFD_CLASS_NAMES[true_label]}")
    axes[1].imshow(pil); axes[1].imshow(cam_b, alpha=0.45, cmap="jet"); axes[1].axis("off")
    axes[1].set_title(f"Before: {SFD_CLASS_NAMES[pred_b]}")
    axes[2].imshow(pil); axes[2].imshow(cam_a, alpha=0.45, cmap="jet"); axes[2].axis("off")
    axes[2].set_title(f"After: {SFD_CLASS_NAMES[pred_a]}")
    plt.tight_layout()

    out = CFG["output_dir"]/ "gradcam" / f"gradcam_{backbone}_{Path(image_path).stem}.png"
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    return out

In [ ]:
# ============================================================
# 21. CANONICAL MASTER TABLE AND REVIEWER CHECKLIST
# ============================================================
def build_canonical_master_table():
    table_dir = CFG["output_dir"] / "tables"
    summaries = []
    for p in table_dir.glob("*_mean_std_ci95_summary.csv"):
        df = pd.read_csv(p)
        summaries.append(df)
    if not summaries:
        print("No summary files found yet.")
        return pd.DataFrame()

    master = pd.concat(summaries, ignore_index=True)
    master.to_csv(table_dir / "ALL_CANONICAL_RESULTS.csv", index=False)

    # Headline values: one source of truth for abstract, tables, and conclusion.
    headline = master[
        ((master.phase=="before") & (master.k==0)) |
        ((master.phase=="after") & (master.k==20))
    ].copy()
    headline.to_csv(table_dir / "HEADLINE_RESULTS_FOR_MANUSCRIPT.csv", index=False)
    return master

def reviewer_checklist():
    items = [
        ("Novelty reframed as protocol/experimental/deployment contribution", "MANUSCRIPT"),
        ("ProtoNet baseline", "CODE+RESULT"),
        ("ANIL baseline", "CODE+RESULT"),
        ("Grad-CAM figure and Results subsection", "CODE+MANUSCRIPT"),
        ("K=1 and K=3", "CODE+RESULT"),
        ("Five support sampling seeds", "CODE+RESULT"),
        ("Mean ± SD and 95% CI", "CODE+RESULT"),
        ("Matched McNemar and paired tests", "CODE+RESULT"),
        ("Per-class before/after metrics", "CODE+RESULT"),
        ("Per-driver metrics", "CODE+RESULT"),
        ("Support/query image overlap audit", "CODE+RESULT"),
        ("Clip/group overlap audit", "CODE+RESULT"),
        ("AUC V1 global negative result and metadata diagnosis", "CODE+RESULT"),
        ("AUC V2 multi-backbone validation", "CODE+RESULT"),
        ("True SFD to AUC V2 transfer", "CODE+RESULT"),
        ("Adaptation time per driver and scope", "CODE+RESULT"),
        ("Latency/FPS/model size", "CODE+RESULT"),
        ("FP16/INT8 claims softened unless actually tested", "MANUSCRIPT"),
        ("Canonical master table for numerical consistency", "CODE+MANUSCRIPT"),
        ("Algorithm and objective explanatory text", "MANUSCRIPT"),
        ("Literature protocol column", "MANUSCRIPT"),
        ("Cautious cross-dataset wording", "MANUSCRIPT"),
    ]
    df = pd.DataFrame(items, columns=["reviewer_requirement","implementation"])
    df.to_csv(CFG["output_dir"]/ "tables" / "Reviewer_Correction_Checklist.csv", index=False)
    return df

# master_table = build_canonical_master_table()
# checklist = reviewer_checklist()

## 22. Recommended Execution Order

1. Set correct paths in `CFG`.
2. Set `CFG["debug"] = True`.
3. Run all definition cells.
4. Run a short SFD debug experiment.
5. Set `CFG["debug"] = False`.
6. Run full SFD experiment.
7. Run AUC V2 within-dataset experiment.
8. Run SFD → AUC V2 cross-dataset experiment.
9. Run AUC V1 global-only negative result.
10. Run ProtoNet and ANIL baselines.
11. Create deployment table.
12. Generate selected Grad-CAM figures.
13. Build the canonical master table and reviewer checklist.

Do not report values until the relevant CSV file has been produced.  
Use `ALL_CANONICAL_RESULTS.csv` as the single source of truth for the manuscript.

# Extended Reviewer Baselines

This extension adds all comparison families explicitly mentioned by the reviewer:

- ProtoNet
- Matching Networks
- Relation Network
- ANIL
- FOMAML
- second-order MAML
- Prototype Memory
- Episodic Memory
- Unified transfer/meta/metric/memory comparison table

In [ ]:
# ============================================================
# EPISODE UTILITIES
# ============================================================
from torch.nn.utils.stateless import functional_call

def sample_episode(df, n_way=10, k_shot=1, q_query=5, seed=11):
    rng = np.random.default_rng(seed)
    classes = sorted(df["label"].unique().tolist())
    chosen = rng.choice(classes, size=min(n_way, len(classes)), replace=False)
    support_parts, query_parts = [], []
    for c in chosen:
        cls = df[df.label == c].reset_index(drop=True)
        if len(cls) < k_shot + 1:
            continue
        idx = np.arange(len(cls))
        rng.shuffle(idx)
        s = cls.iloc[idx[:min(k_shot, len(cls)-1)]]
        q = cls.iloc[idx[len(s):len(s)+min(q_query, len(cls)-len(s))]]
        support_parts.append(s)
        query_parts.append(q)
    if not support_parts or not query_parts:
        raise ValueError("Unable to build a valid episode.")
    return pd.concat(support_parts, ignore_index=True), pd.concat(query_parts, ignore_index=True)

def loader_to_tensors(loader):
    xs, ys = [], []
    for x, y in loader:
        xs.append(x.to(DEVICE))
        ys.append(y.to(DEVICE))
    return torch.cat(xs), torch.cat(ys)

def extract_features(extractor, df):
    loader = make_loader(df, eval_tfms, shuffle=False)
    zs, ys = [], []
    extractor.eval()
    with torch.no_grad():
        for x, y in loader:
            zs.append(extractor(x.to(DEVICE)).cpu())
            ys.append(y.cpu())
    return torch.cat(zs), torch.cat(ys)

In [ ]:
# ============================================================
# MATCHING NETWORK
# ============================================================
def matching_network_predict(extractor, support_df, query_df, temperature=1.0):
    sz, sy = extract_features(extractor, support_df)
    qz, qy = extract_features(extractor, query_df)
    sz = F.normalize(sz.to(DEVICE), dim=1)
    qz = F.normalize(qz.to(DEVICE), dim=1)
    attn = torch.softmax((qz @ sz.T) / temperature, dim=1)
    probs = attn @ F.one_hot(sy.to(DEVICE), num_classes=10).float()
    return qy.numpy(), probs.argmax(1).cpu().numpy()

def run_matchingnet_baseline(dataset_name, test_df, backbone="resnet50"):
    extractor = FeatureExtractor(backbone).to(DEVICE)
    rows = []
    for k in [1,3,5,10,20]:
        for seed in CFG["seeds"]:
            yt, yp = [], []
            for driver, ddf in test_df.groupby("driver"):
                support, query = support_query_split(ddf, k, seed, strict_group=True)
                if support.empty or query.empty:
                    continue
                y, p = matching_network_predict(extractor, support, query)
                yt.extend(y); yp.extend(p)
            rows.append({"dataset":dataset_name,"method":"MatchingNet","backbone":backbone,
                         "k":k,"seed":seed,"accuracy":accuracy_score(yt,yp),
                         "macro_f1":f1_score(yt,yp,average="macro",zero_division=0)})
    out = pd.DataFrame(rows)
    out.to_csv(CFG["output_dir"]/ "tables" / f"{dataset_name}_MatchingNet_baseline.csv", index=False)
    return out

In [ ]:
# ============================================================
# RELATION NETWORK
# ============================================================
class RelationModule(nn.Module):
    def __init__(self, feature_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(feature_dim*2, hidden), nn.ReLU(), nn.Linear(hidden,1))
    def forward(self, q, p):
        q = q.unsqueeze(1).expand(-1,p.size(0),-1)
        p = p.unsqueeze(0).expand(q.size(0),-1,-1)
        return self.net(torch.cat([q,p],dim=-1)).squeeze(-1)

def relationnet_predict(extractor, support_df, query_df, epochs=20, lr=1e-3):
    sz, sy = extract_features(extractor, support_df)
    qz, qy = extract_features(extractor, query_df)
    sz, sy, qz, qy = sz.to(DEVICE), sy.to(DEVICE), qz.to(DEVICE), qy.to(DEVICE)
    classes = sorted(sy.unique().tolist())
    protos = torch.stack([sz[sy==c].mean(0) for c in classes])
    mapper = {c:i for i,c in enumerate(classes)}
    targets = torch.tensor([mapper[int(y)] for y in qy], device=DEVICE)
    relation = RelationModule(extractor.out_dim).to(DEVICE)
    opt = torch.optim.Adam(relation.parameters(), lr=lr)
    for _ in range(epochs):
        loss = F.cross_entropy(relation(qz, protos), targets)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        idx = relation(qz, protos).argmax(1).cpu().tolist()
    return qy.cpu().numpy(), np.array([classes[i] for i in idx])

def run_relationnet_baseline(dataset_name, test_df, backbone="resnet50"):
    extractor = FeatureExtractor(backbone).to(DEVICE)
    rows = []
    for k in [1,3,5,10,20]:
        for seed in CFG["seeds"]:
            yt, yp = [], []
            for driver, ddf in test_df.groupby("driver"):
                support, query = support_query_split(ddf, k, seed, strict_group=True)
                if support.empty or query.empty:
                    continue
                y,p = relationnet_predict(extractor,support,query)
                yt.extend(y); yp.extend(p)
            rows.append({"dataset":dataset_name,"method":"RelationNet","backbone":backbone,
                         "k":k,"seed":seed,"accuracy":accuracy_score(yt,yp),
                         "macro_f1":f1_score(yt,yp,average="macro",zero_division=0)})
    out=pd.DataFrame(rows)
    out.to_csv(CFG["output_dir"]/ "tables" / f"{dataset_name}_RelationNet_baseline.csv",index=False)
    return out

In [ ]:
# ============================================================
# FOMAML AND SECOND-ORDER MAML
# ============================================================
class MetaLinearClassifier(nn.Module):
    def __init__(self, in_dim, num_classes=10):
        super().__init__()
        self.linear = nn.Linear(in_dim,num_classes)
    def forward(self,x):
        return self.linear(x)

def inner_update(model, params, x, y, lr, create_graph):
    logits = functional_call(model, params, (x,))
    loss = F.cross_entropy(logits,y)
    grads = torch.autograd.grad(loss, tuple(params.values()), create_graph=create_graph)
    return {n:p-lr*g for (n,p),g in zip(params.items(),grads)}

def meta_train_classifier(extractor, train_df, method="MAML", meta_epochs=20,
                          episodes_per_epoch=20, inner_steps=3, inner_lr=0.01,
                          outer_lr=1e-3, seed=11):
    set_seed(seed)
    model = MetaLinearClassifier(extractor.out_dim,10).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(),lr=outer_lr)
    second_order = method.upper()=="MAML"
    hist=[]
    for epoch in range(1,meta_epochs+1):
        opt.zero_grad()
        meta_loss=0.0; valid=0
        for ep in range(episodes_per_epoch):
            try:
                s_df,q_df=sample_episode(train_df,k_shot=1,q_query=5,seed=seed+epoch*1000+ep)
            except ValueError:
                continue
            sx,sy=loader_to_tensors(make_loader(s_df,eval_tfms,batch_size=len(s_df)))
            qx,qy=loader_to_tensors(make_loader(q_df,eval_tfms,batch_size=len(q_df)))
            with torch.no_grad():
                sf=extractor(sx); qf=extractor(qx)
            params={n:p for n,p in model.named_parameters()}
            for _ in range(inner_steps):
                params=inner_update(model,params,sf,sy,inner_lr,create_graph=second_order)
            meta_loss = meta_loss + F.cross_entropy(functional_call(model,params,(qf,)),qy)
            valid += 1
        meta_loss = meta_loss/max(valid,1)
        meta_loss.backward(); opt.step()
        hist.append({"epoch":epoch,"meta_loss":float(meta_loss.detach().cpu())})
        print(method,epoch,hist[-1]["meta_loss"])
    return model,pd.DataFrame(hist)

def meta_predict(extractor, meta_model, support_df, query_df, second_order=False):
    sx,sy=loader_to_tensors(make_loader(support_df,eval_tfms,batch_size=len(support_df)))
    qx,qy=loader_to_tensors(make_loader(query_df,eval_tfms,batch_size=len(query_df)))
    with torch.no_grad():
        sf=extractor(sx); qf=extractor(qx)
    params={n:p for n,p in meta_model.named_parameters()}
    for _ in range(5):
        params=inner_update(meta_model,params,sf,sy,0.01,create_graph=second_order)
    with torch.no_grad():
        pred=functional_call(meta_model,params,(qf,)).argmax(1).cpu().numpy()
    return qy.cpu().numpy(),pred

def run_maml_family_baseline(dataset_name,train_df,test_df,backbone="resnet50",method="FOMAML"):
    extractor=FeatureExtractor(backbone).to(DEVICE)
    model,hist=meta_train_classifier(extractor,train_df,method=method,
        meta_epochs=5 if CFG["debug"] else 20,
        episodes_per_epoch=5 if CFG["debug"] else 20,
        seed=CFG["seeds"][0])
    hist.to_csv(CFG["output_dir"]/ "tables" / f"{dataset_name}_{method}_meta_history.csv",index=False)
    rows=[]
    for k in [1,3,5,10,20]:
        for seed in CFG["seeds"]:
            yt,yp=[],[]
            for driver,ddf in test_df.groupby("driver"):
                support,query=support_query_split(ddf,k,seed,strict_group=True)
                if support.empty or query.empty: continue
                y,p=meta_predict(extractor,model,support,query,second_order=(method.upper()=="MAML"))
                yt.extend(y); yp.extend(p)
            rows.append({"dataset":dataset_name,"method":method,"backbone":backbone,
                         "k":k,"seed":seed,"accuracy":accuracy_score(yt,yp),
                         "macro_f1":f1_score(yt,yp,average="macro",zero_division=0)})
    out=pd.DataFrame(rows)
    out.to_csv(CFG["output_dir"]/ "tables" / f"{dataset_name}_{method}_baseline.csv",index=False)
    return out

In [ ]:
# ============================================================
# PROTOTYPE MEMORY AND EPISODIC MEMORY
# ============================================================
class PrototypeMemory:
    def __init__(self,momentum=0.9):
        self.momentum=momentum; self.memory={}
    def update(self,z,y):
        for c in y.unique().tolist():
            p=z[y==c].mean(0)
            self.memory[c]=p if c not in self.memory else self.momentum*self.memory[c]+(1-self.momentum)*p
    def predict(self,q):
        classes=sorted(self.memory)
        protos=torch.stack([self.memory[c] for c in classes]).to(q.device)
        idx=torch.cdist(q,protos).argmin(1).cpu().tolist()
        return np.array([classes[i] for i in idx])

class EpisodicMemoryBank:
    def __init__(self,max_items_per_class=50):
        self.max_items_per_class=max_items_per_class
        self.bank=defaultdict(list)
    def add(self,z,y):
        for feat,lab in zip(z,y):
            c=int(lab); self.bank[c].append(feat.detach().cpu())
            self.bank[c]=self.bank[c][-self.max_items_per_class:]
    def predict(self,q,top_k=5):
        feats=[]; labels=[]
        for c,items in self.bank.items():
            feats.extend(items); labels.extend([c]*len(items))
        m=F.normalize(torch.stack(feats).to(q.device),dim=1)
        q=F.normalize(q,dim=1)
        labels=torch.tensor(labels,device=q.device)
        idx=(q@m.T).topk(k=min(top_k,m.size(0)),dim=1).indices
        nn_labels=labels[idx]
        preds=[]
        for row in nn_labels:
            vals,counts=row.unique(return_counts=True)
            preds.append(int(vals[counts.argmax()]))
        return np.array(preds)

def memory_predict(extractor,support_df,query_df,method="PrototypeMemory"):
    sz,sy=extract_features(extractor,support_df)
    qz,qy=extract_features(extractor,query_df)
    if method=="PrototypeMemory":
        mem=PrototypeMemory(); mem.update(sz,sy); preds=mem.predict(qz.to(DEVICE))
    else:
        mem=EpisodicMemoryBank(); mem.add(sz,sy); preds=mem.predict(qz.to(DEVICE))
    return qy.numpy(),preds

def run_memory_baseline(dataset_name,test_df,backbone="resnet50",method="PrototypeMemory"):
    extractor=FeatureExtractor(backbone).to(DEVICE)
    rows=[]
    for k in [1,3,5,10,20]:
        for seed in CFG["seeds"]:
            yt,yp=[],[]
            for driver,ddf in test_df.groupby("driver"):
                support,query=support_query_split(ddf,k,seed,strict_group=True)
                if support.empty or query.empty: continue
                y,p=memory_predict(extractor,support,query,method)
                yt.extend(y); yp.extend(p)
            rows.append({"dataset":dataset_name,"method":method,"backbone":backbone,
                         "k":k,"seed":seed,"accuracy":accuracy_score(yt,yp),
                         "macro_f1":f1_score(yt,yp,average="macro",zero_division=0)})
    out=pd.DataFrame(rows)
    out.to_csv(CFG["output_dir"]/ "tables" / f"{dataset_name}_{method}_baseline.csv",index=False)
    return out

In [ ]:
# ============================================================
# UNIFIED COMPARISON TABLE
# ============================================================
METHOD_METADATA={
"HeadFineTuning":("Transfer learning",False,False,False,False),
"LastBlockFineTuning":("Transfer learning",False,False,False,False),
"FullFineTuning":("Transfer learning",False,False,False,False),
"ProtoNet":("Metric meta-learning",True,True,False,True),
"MatchingNet":("Metric meta-learning",True,True,False,True),
"RelationNet":("Metric meta-learning",True,True,False,False),
"ANIL":("Gradient meta-learning",True,False,False,False),
"FOMAML":("Gradient meta-learning",True,False,False,False),
"MAML":("Gradient meta-learning",True,False,True,False),
"PrototypeMemory":("Memory-based",False,True,False,True),
"EpisodicMemory":("Memory-based",False,True,False,True),
}

def summarize_baseline_df(df):
    method=str(df.method.iloc[0])
    family,meta,metric,second,memory=METHOD_METADATA.get(method,("",False,False,False,False))
    rows=[]
    for k,g in df.groupby("k"):
        alo,ahi=ci95(g.accuracy.values); flo,fhi=ci95(g.macro_f1.values)
        rows.append({"method":method,"family":family,"k":k,"n_seeds":g.seed.nunique(),
                     "accuracy_mean":g.accuracy.mean(),"accuracy_std":g.accuracy.std(ddof=1),
                     "accuracy_ci95_low":alo,"accuracy_ci95_high":ahi,
                     "macro_f1_mean":g.macro_f1.mean(),"macro_f1_std":g.macro_f1.std(ddof=1),
                     "macro_f1_ci95_low":flo,"macro_f1_ci95_high":fhi,
                     "meta_learning":meta,"metric_learning":metric,
                     "second_order":second,"memory_module":memory})
    return pd.DataFrame(rows)

def build_unified_meta_memory_comparison(dataset_name="SFD"):
    tdir=CFG["output_dir"]/ "tables"
    files=[
      f"{dataset_name}_protonet_baseline.csv",
      f"{dataset_name}_ANIL_baseline.csv",
      f"{dataset_name}_MatchingNet_baseline.csv",
      f"{dataset_name}_RelationNet_baseline.csv",
      f"{dataset_name}_FOMAML_baseline.csv",
      f"{dataset_name}_MAML_baseline.csv",
      f"{dataset_name}_PrototypeMemory_baseline.csv",
      f"{dataset_name}_EpisodicMemory_baseline.csv",
    ]
    frames=[]
    for name in files:
        p=tdir/name
        if p.exists():
            frames.append(summarize_baseline_df(pd.read_csv(p)))
    summary_path=tdir/f"{dataset_name}_mean_std_ci95_summary.csv"
    if summary_path.exists():
        ft=pd.read_csv(summary_path)
        ft=ft[ft.phase=="after"].copy()
        ft["method"]=ft.scope.map({"head":"HeadFineTuning","last_block":"LastBlockFineTuning","full":"FullFineTuning"})
        ft["family"]="Transfer learning"; ft["meta_learning"]=False; ft["metric_learning"]=False
        ft["second_order"]=False; ft["memory_module"]=False
        frames.append(ft[["method","family","k","n_seeds","accuracy_mean","accuracy_std",
                          "accuracy_ci95_low","accuracy_ci95_high","macro_f1_mean","macro_f1_std",
                          "macro_f1_ci95_low","macro_f1_ci95_high","meta_learning",
                          "metric_learning","second_order","memory_module"]])
    if not frames:
        return pd.DataFrame()
    out=pd.concat(frames,ignore_index=True)
    out.to_csv(tdir/f"{dataset_name}_UNIFIED_META_MEMORY_COMPARISON.csv",index=False)
    return out

## Run all added baselines

```python
protonet_sfd = run_protonet_baseline("SFD", sfd_test_df)
anil_sfd = run_anil_baseline("SFD", sfd_test_df)
matchingnet_sfd = run_matchingnet_baseline("SFD", sfd_test_df)
relationnet_sfd = run_relationnet_baseline("SFD", sfd_test_df)

fomaml_sfd = run_maml_family_baseline(
    "SFD", sfd_train_df, sfd_test_df, method="FOMAML"
)

maml_sfd = run_maml_family_baseline(
    "SFD", sfd_train_df, sfd_test_df, method="MAML"
)

prototype_memory_sfd = run_memory_baseline(
    "SFD", sfd_test_df, method="PrototypeMemory"
)

episodic_memory_sfd = run_memory_baseline(
    "SFD", sfd_test_df, method="EpisodicMemory"
)

unified_comparison = build_unified_meta_memory_comparison("SFD")
```

Run FOMAML before full second-order MAML. Full MAML is the most computationally expensive baseline.

## Required manuscript positioning

Even after adding all baselines, the proposed method must be described accurately:

> The proposed method uses standard supervised fine-tuning on a small labeled support set. ProtoNet, MatchingNet, RelationNet, ANIL, FOMAML, MAML, prototype memory, and episodic memory are comparison baselines. The paper's primary contribution is protocol-based, experimental, statistical, and deployment-oriented rather than a new meta-learning optimization algorithm.

# Final Execution Controller

The earlier cells only define functions. Run the controller below to start the experiments.

For a quick test, set `RUN_MODE = "debug"`.  
For final reviewer results, set `RUN_MODE = "full"`.

Full MAML is computationally expensive, so it can be enabled separately.

In [ ]:
# ============================================================
# FINAL EXECUTION CONTROLLER
# ============================================================
RUN_MODE = "debug"   # change to "full" for final experiments

RUN_SFD_MAIN = True
RUN_AUCV2_WITHIN = False
RUN_CROSS_DATASET = False
RUN_AUCV1_GLOBAL = False

RUN_PROTONET = False
RUN_ANIL = False
RUN_MATCHINGNET = False
RUN_RELATIONNET = False
RUN_FOMAML = False
RUN_MAML = False
RUN_PROTOTYPE_MEMORY = False
RUN_EPISODIC_MEMORY = False

RUN_DEPLOYMENT_PROFILE = False
RUN_FINAL_TABLES = True

CFG["debug"] = (RUN_MODE == "debug")

print("=" * 80)
print("FINAL EXECUTION CONTROLLER")
print("=" * 80)
print("Mode:", RUN_MODE)
print("Device:", DEVICE)
print("Output folder:", CFG["output_dir"].resolve())
print("Debug:", CFG["debug"])
print("=" * 80)

completed = []

if RUN_SFD_MAIN:
    print("\n[1] Running SFD main personalization experiment...")
    sfd_pred, sfd_timing, sfd_audit = run_personalization_experiment(
        "SFD", sfd_train_df, sfd_test_df, CFG["backbones"]
    )
    sfd_summary, sfd_class, sfd_driver, sfd_paired, sfd_mcnemar = save_all_analysis(
        "SFD", sfd_pred
    )
    completed.append("SFD main experiment")

if RUN_AUCV2_WITHIN:
    print("\n[2] Running AUC V2 within-dataset multi-backbone experiment...")
    aucv2_pred, aucv2_timing, aucv2_audit = run_personalization_experiment(
        "AUCV2_WITHIN", auc_v2_train_df, auc_v2_test_df, CFG["backbones"]
    )
    aucv2_summary, aucv2_class, aucv2_driver, aucv2_paired, aucv2_mcnemar = save_all_analysis(
        "AUCV2_WITHIN", aucv2_pred
    )
    completed.append("AUC V2 within-dataset experiment")

if RUN_CROSS_DATASET:
    print("\n[3] Running true SFD -> AUC V2 cross-dataset validation...")
    cross_pred = run_cross_dataset_sfd_to_aucv2()
    completed.append("SFD -> AUC V2 cross-dataset experiment")

if RUN_AUCV1_GLOBAL:
    print("\n[4] Running AUC V1 global-only negative-result analysis...")
    auc_v1_global_result = evaluate_auc_v1_global_only()
    display(auc_v1_global_result)
    completed.append("AUC V1 global-only analysis")

if RUN_PROTONET:
    print("\n[5] Running ProtoNet baseline...")
    protonet_sfd = run_protonet_baseline("SFD", sfd_test_df)
    completed.append("ProtoNet")

if RUN_ANIL:
    print("\n[6] Running ANIL baseline...")
    anil_sfd = run_anil_baseline("SFD", sfd_test_df)
    completed.append("ANIL")

if RUN_MATCHINGNET:
    print("\n[7] Running MatchingNet baseline...")
    matchingnet_sfd = run_matchingnet_baseline("SFD", sfd_test_df)
    completed.append("MatchingNet")

if RUN_RELATIONNET:
    print("\n[8] Running RelationNet baseline...")
    relationnet_sfd = run_relationnet_baseline("SFD", sfd_test_df)
    completed.append("RelationNet")

if RUN_FOMAML:
    print("\n[9] Running FOMAML baseline...")
    fomaml_sfd = run_maml_family_baseline(
        "SFD", sfd_train_df, sfd_test_df, method="FOMAML"
    )
    completed.append("FOMAML")

if RUN_MAML:
    print("\n[10] Running second-order MAML baseline...")
    maml_sfd = run_maml_family_baseline(
        "SFD", sfd_train_df, sfd_test_df, method="MAML"
    )
    completed.append("MAML")

if RUN_PROTOTYPE_MEMORY:
    print("\n[11] Running Prototype Memory baseline...")
    prototype_memory_sfd = run_memory_baseline(
        "SFD", sfd_test_df, method="PrototypeMemory"
    )
    completed.append("Prototype Memory")

if RUN_EPISODIC_MEMORY:
    print("\n[12] Running Episodic Memory baseline...")
    episodic_memory_sfd = run_memory_baseline(
        "SFD", sfd_test_df, method="EpisodicMemory"
    )
    completed.append("Episodic Memory")

if RUN_DEPLOYMENT_PROFILE:
    print("\n[13] Running latency/FPS/model-size profiling...")
    deployment_table = create_deployment_table()
    display(deployment_table)
    completed.append("Deployment profiling")

if RUN_FINAL_TABLES:
    print("\n[14] Building final canonical and comparison tables...")
    master_table = build_canonical_master_table()
    checklist = reviewer_checklist()
    unified_comparison = build_unified_meta_memory_comparison("SFD")
    if not master_table.empty:
        display(master_table.head())
    if not unified_comparison.empty:
        display(unified_comparison.head())
    completed.append("Final tables")

print("\n" + "=" * 80)
print("COMPLETED")
print("=" * 80)
for item in completed:
    print("✓", item)

print("\nGenerated files are inside:")
print(CFG["output_dir"].resolve())

print("\nImportant folders:")
for sub in ["tables", "figures", "checkpoints", "predictions", "audits", "gradcam"]:
    p = CFG["output_dir"] / sub
    print("-", p.resolve(), "| files:", len(list(p.glob("*"))))

print("\nFor the complete reviewer run:")
print('1. Set RUN_MODE = "full"')
print("2. Enable the required RUN_* options")
print("3. Run this controller cell again")